# NOTE - My modeling scripts basically do what I want for the time being, but there may be data cleaning issues especially with county names. Also have not added demographic, education etc to this. Other notebook has census, copy it over here.

# Also have to implement much more rigorous validation, and try other models.

In [1]:
import pandas as pd

In [2]:
elections = pd.read_csv('/Users/matthewcavanaugh/Desktop/GitHub/us-presidential-elections-by-county/sources/countypres_2000-2020.csv')
elections

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,AL GORE,DEMOCRAT,4942,17208,20220315,TOTAL
1,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,11993,17208,20220315,TOTAL
2,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,RALPH NADER,GREEN,160,17208,20220315,TOTAL
3,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,OTHER,OTHER,113,17208,20220315,TOTAL
4,2000,ALABAMA,AL,BALDWIN,1003.0,US PRESIDENT,AL GORE,DEMOCRAT,13997,56480,20220315,TOTAL
...,...,...,...,...,...,...,...,...,...,...,...,...
72612,2020,WYOMING,WY,WASHAKIE,56043.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,3245,4032,20220315,TOTAL
72613,2020,WYOMING,WY,WESTON,56045.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,360,3560,20220315,TOTAL
72614,2020,WYOMING,WY,WESTON,56045.0,US PRESIDENT,JO JORGENSEN,LIBERTARIAN,46,3560,20220315,TOTAL
72615,2020,WYOMING,WY,WESTON,56045.0,US PRESIDENT,OTHER,OTHER,47,3560,20220315,TOTAL


EDA

In [3]:
elections.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72617 entries, 0 to 72616
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   year            72617 non-null  int64  
 1   state           72617 non-null  object 
 2   state_po        72617 non-null  object 
 3   county_name     72617 non-null  object 
 4   county_fips     72565 non-null  float64
 5   office          72617 non-null  object 
 6   candidate       72617 non-null  object 
 7   party           72617 non-null  object 
 8   candidatevotes  72617 non-null  int64  
 9   totalvotes      72617 non-null  int64  
 10  version         72617 non-null  int64  
 11  mode            72617 non-null  object 
dtypes: float64(1), int64(4), object(7)
memory usage: 6.6+ MB


In [4]:
elections.isnull().sum()

year               0
state              0
state_po           0
county_name        0
county_fips       52
office             0
candidate          0
party              0
candidatevotes     0
totalvotes         0
version            0
mode               0
dtype: int64

Where are the nulls?

### Here we have examples of unusual reporting statistics.

In [5]:
elections[elections['county_fips'].isnull()]

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
12452,2000,CONNECTICUT,CT,STATEWIDE WRITEIN,NaN,US PRESIDENT,AL GORE,DEMOCRAT,0,0,20220315,TOTAL
12453,2000,MAINE,ME,MAINE UOCAVA,NaN,US PRESIDENT,AL GORE,DEMOCRAT,0,0,20220315,TOTAL
12454,2000,RHODE ISLAND,RI,FEDERAL PRECINCT,NaN,US PRESIDENT,AL GORE,DEMOCRAT,0,0,20220315,TOTAL
12455,2000,CONNECTICUT,CT,STATEWIDE WRITEIN,NaN,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,0,0,20220315,TOTAL
12456,2000,MAINE,ME,MAINE UOCAVA,NaN,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,0,0,20220315,TOTAL
12457,2000,RHODE ISLAND,RI,FEDERAL PRECINCT,NaN,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,0,0,20220315,TOTAL
12458,2000,CONNECTICUT,CT,STATEWIDE WRITEIN,NaN,US PRESIDENT,RALPH NADER,GREEN,0,0,20220315,TOTAL
12459,2000,MAINE,ME,MAINE UOCAVA,NaN,US PRESIDENT,RALPH NADER,GREEN,0,0,20220315,TOTAL
12460,2000,RHODE ISLAND,RI,FEDERAL PRECINCT,NaN,US PRESIDENT,RALPH NADER,GREEN,0,0,20220315,TOTAL
12461,2000,CONNECTICUT,CT,STATEWIDE WRITEIN,NaN,US PRESIDENT,OTHER,OTHER,0,0,20220315,TOTAL


These are unusual administrative units that have no votes recorded. It's safe to drop them.

In [6]:
elections.dropna(inplace=True)

In [7]:
# Combine 'County' and 'State' columns
#elections['county_name'] = elections['county_name'] + ', ' + elections['state']

#elections.drop('state', axis=1, inplace=True)
elections.drop('state_po', axis=1, inplace=True)
elections.drop('county_fips', axis=1, inplace=True)
elections.drop('office', axis=1, inplace=True)
elections.drop('version', axis=1, inplace=True)
elections.drop('mode', axis=1, inplace=True)
elections.drop('candidate', axis=1, inplace=True)

# Result
elections

,year,state,county_name,party,candidatevotes,totalvotes
0,2000,ALABAMA,AUTAUGA,DEMOCRAT,4942,17208
1,2000,ALABAMA,AUTAUGA,REPUBLICAN,11993,17208
2,2000,ALABAMA,AUTAUGA,GREEN,160,17208
3,2000,ALABAMA,AUTAUGA,OTHER,113,17208
4,2000,ALABAMA,BALDWIN,DEMOCRAT,13997,56480
...,...,...,...,...,...,...
72612,2020,WYOMING,WASHAKIE,REPUBLICAN,3245,4032
72613,2020,WYOMING,WESTON,DEMOCRAT,360,3560
72614,2020,WYOMING,WESTON,LIBERTARIAN,46,3560
72615,2020,WYOMING,WESTON,OTHER,47,3560


In [8]:
elections = elections.rename(columns={'year': 'Year'})
elections = elections.rename(columns={'state': 'State'})
elections = elections.rename(columns={'county_name': 'County'})
elections = elections.rename(columns={'state': 'State'})
elections = elections.rename(columns={'party': 'Party'})
elections = elections.rename(columns={'candidatevotes': 'Votes'})
elections = elections.rename(columns={'totalvotes': 'Total'})

elections

,Year,State,County,Party,Votes,Total
0,2000,ALABAMA,AUTAUGA,DEMOCRAT,4942,17208
1,2000,ALABAMA,AUTAUGA,REPUBLICAN,11993,17208
2,2000,ALABAMA,AUTAUGA,GREEN,160,17208
3,2000,ALABAMA,AUTAUGA,OTHER,113,17208
4,2000,ALABAMA,BALDWIN,DEMOCRAT,13997,56480
...,...,...,...,...,...,...
72612,2020,WYOMING,WASHAKIE,REPUBLICAN,3245,4032
72613,2020,WYOMING,WESTON,DEMOCRAT,360,3560
72614,2020,WYOMING,WESTON,LIBERTARIAN,46,3560
72615,2020,WYOMING,WESTON,OTHER,47,3560


In [9]:
def clean_text(text):
    if isinstance(text, str):
        return text.title()  # Capitalizes the first letter of each word and makes the rest lowercase
    return text  # Return text unchanged if it's not a string

# Apply this function to a specific column in your dataframe
elections['County'] = elections['County'].apply(clean_text)
elections['Party'] = elections['Party'].apply(clean_text)
elections['State'] = elections['State'].apply(clean_text)

In [10]:
elections

,Year,State,County,Party,Votes,Total
0,2000,Alabama,Autauga,Democrat,4942,17208
1,2000,Alabama,Autauga,Republican,11993,17208
2,2000,Alabama,Autauga,Green,160,17208
3,2000,Alabama,Autauga,Other,113,17208
4,2000,Alabama,Baldwin,Democrat,13997,56480
...,...,...,...,...,...,...
72612,2020,Wyoming,Washakie,Republican,3245,4032
72613,2020,Wyoming,Weston,Democrat,360,3560
72614,2020,Wyoming,Weston,Libertarian,46,3560
72615,2020,Wyoming,Weston,Other,47,3560


Virginia is an unusual case with many independent cities. Let's check it to ensure the capitlization has worked the way we want.

In [11]:
elections[elections['State'].str.contains('Virginia')]

,Year,State,County,Party,Votes,Total
11160,2000,Virginia,Accomack,Democrat,5092,11925
11161,2000,Virginia,Accomack,Republican,6352,11925
11162,2000,Virginia,Accomack,Green,220,11925
11163,2000,Virginia,Accomack,Other,261,11925
11164,2000,Virginia,Albemarle,Democrat,16255,36846
...,...,...,...,...,...,...
71965,2020,Virginia,Winchester City,Other,24,12113
71966,2020,Virginia,Winchester City,Other,0,12113
71967,2020,Virginia,Winchester City,Republican,2825,12113
71968,2020,Virginia,Winchester City,Republican,2370,12113


We may have no choice but to drop Alaska as it does not have consistent administrative units by which metrics are kept.

In [12]:
elections = elections[~elections['State'].str.contains('Alaska', na=False)]

In [13]:
elections['Party'].value_counts()

Party
Democrat       20644
Republican     20644
Other          19553
Green           5950
Libertarian     4913
Name: count, dtype: int64

In [14]:
elections['Percentage'] = round((elections['Votes'] / elections['Total'] * 100), 2)
elections

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_95429/3329014358.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  elections['Percentage'] = round((elections['Votes'] / elections['Total'] * 100), 2)


,Year,State,County,Party,Votes,Total,Percentage
0,2000,Alabama,Autauga,Democrat,4942,17208,28.72
1,2000,Alabama,Autauga,Republican,11993,17208,69.69
2,2000,Alabama,Autauga,Green,160,17208,0.93
3,2000,Alabama,Autauga,Other,113,17208,0.66
4,2000,Alabama,Baldwin,Democrat,13997,56480,24.78
...,...,...,...,...,...,...,...
72612,2020,Wyoming,Washakie,Republican,3245,4032,80.48
72613,2020,Wyoming,Weston,Democrat,360,3560,10.11
72614,2020,Wyoming,Weston,Libertarian,46,3560,1.29
72615,2020,Wyoming,Weston,Other,47,3560,1.32


In [15]:
elections = elections[['Year', 'State', 'County', 'Party', 'Votes', 'Percentage', 'Total']]
elections

,Year,State,County,Party,Votes,Percentage,Total
0,2000,Alabama,Autauga,Democrat,4942,28.72,17208
1,2000,Alabama,Autauga,Republican,11993,69.69,17208
2,2000,Alabama,Autauga,Green,160,0.93,17208
3,2000,Alabama,Autauga,Other,113,0.66,17208
4,2000,Alabama,Baldwin,Democrat,13997,24.78,56480
...,...,...,...,...,...,...,...
72612,2020,Wyoming,Washakie,Republican,3245,80.48,4032
72613,2020,Wyoming,Weston,Democrat,360,10.11,3560
72614,2020,Wyoming,Weston,Libertarian,46,1.29,3560
72615,2020,Wyoming,Weston,Other,47,1.32,3560


In [16]:
# Pivot the DataFrame for vote totals by year and county for each party
vote_totals = elections.pivot_table(index=['Year', 'State', 'County'], columns='Party', values='Votes', aggfunc='sum')

# Pivot the DataFrame for percentages by year and county for each party
percentage_totals = elections.pivot_table(index=['Year', 'State', 'County'], columns='Party', values='Percentage', aggfunc='sum')

# Merge the two DataFrames together: vote totals and percentages
election_pivot = pd.concat([vote_totals, percentage_totals], axis=1, keys=['Votes', 'Percentage'])

# Flatten the MultiIndex columns and rename them according to the required format
election_pivot.columns = [f'{party} {stat}' for stat, party in election_pivot.columns]

# Reorder columns to keep the parties together (Democratic, Republican, etc.)
party_order = ['Democrat', 'Republican', 'Libertarian', 'Green', 'Other']  # You can add or adjust parties as needed
ordered_columns = []

for party in party_order:
    if f'{party} Votes' in election_pivot.columns:
        ordered_columns.append(f'{party} Votes')
    if f'{party} Percentage' in election_pivot.columns:
        ordered_columns.append(f'{party} Percentage')

# Reindex the DataFrame to match the desired column order
election_pivot = election_pivot[ordered_columns]

# Include the 'Total' column in the final DataFrame
election_results_totals = elections[['Year', 'State', 'County', 'Total']].drop_duplicates()

# Merge the 'Total' column back into the pivoted DataFrame
election_pivot = pd.merge(election_results_totals, election_pivot, on=['Year', 'State', 'County'])

# Move the 'Total' column to the end
election_pivot = election_pivot[[col for col in election_pivot.columns if col != 'Total'] + ['Total']]

# Fill nulls with 0
election_pivot.fillna(0, inplace=True)

# Display the pivoted DataFrame with the correct column ordering and Total column at the end
election_pivot

,Year,State,County,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Libertarian Votes,Libertarian Percentage,Green Votes,Green Percentage,Other Votes,Other Percentage,Total
0,2000,Alabama,Autauga,4942.0,28.72,11993.0,69.69,0.0,0.00,160.0,0.93,113.0,0.66,17208
1,2000,Alabama,Baldwin,13997.0,24.78,40872.0,72.37,0.0,0.00,1033.0,1.83,578.0,1.02,56480
2,2000,Alabama,Barbour,5188.0,49.91,5096.0,49.02,0.0,0.00,46.0,0.44,65.0,0.63,10395
3,2000,Alabama,Bibb,2710.0,38.16,4273.0,60.17,0.0,0.00,52.0,0.73,66.0,0.93,7101
4,2000,Alabama,Blount,4977.0,27.69,12667.0,70.48,0.0,0.00,154.0,0.86,175.0,0.97,17973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18677,2020,Wyoming,Sweetwater,3823.0,22.89,12229.0,73.24,350.0,2.10,0.0,0.00,296.0,1.77,16698
18678,2020,Wyoming,Teton,9848.0,66.60,4341.0,29.36,255.0,1.72,0.0,0.00,343.0,2.32,14787
18679,2020,Wyoming,Uinta,1591.0,16.82,7496.0,79.25,172.0,1.82,0.0,0.00,200.0,2.11,9459
18680,2020,Wyoming,Washakie,651.0,16.15,3245.0,80.48,65.0,1.61,0.0,0.00,71.0,1.76,4032


In [17]:
election_pivot['D-R margin'] = election_pivot['Democrat Percentage'] - election_pivot['Republican Percentage']

In [18]:
election_pivot['Two party vote share'] = election_pivot['Democrat Percentage'] + election_pivot['Republican Percentage']

In [19]:
# Move the 'Total' column to the end
election_pivot = election_pivot[[col for col in election_pivot.columns if col != 'Total'] + ['Total']]

In [20]:
election_pivot

,Year,State,County,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Libertarian Votes,Libertarian Percentage,Green Votes,Green Percentage,Other Votes,Other Percentage,D-R margin,Two party vote share,Total
0,2000,Alabama,Autauga,4942.0,28.72,11993.0,69.69,0.0,0.00,160.0,0.93,113.0,0.66,-40.97,98.41,17208
1,2000,Alabama,Baldwin,13997.0,24.78,40872.0,72.37,0.0,0.00,1033.0,1.83,578.0,1.02,-47.59,97.15,56480
2,2000,Alabama,Barbour,5188.0,49.91,5096.0,49.02,0.0,0.00,46.0,0.44,65.0,0.63,0.89,98.93,10395
3,2000,Alabama,Bibb,2710.0,38.16,4273.0,60.17,0.0,0.00,52.0,0.73,66.0,0.93,-22.01,98.33,7101
4,2000,Alabama,Blount,4977.0,27.69,12667.0,70.48,0.0,0.00,154.0,0.86,175.0,0.97,-42.79,98.17,17973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18677,2020,Wyoming,Sweetwater,3823.0,22.89,12229.0,73.24,350.0,2.10,0.0,0.00,296.0,1.77,-50.35,96.13,16698
18678,2020,Wyoming,Teton,9848.0,66.60,4341.0,29.36,255.0,1.72,0.0,0.00,343.0,2.32,37.24,95.96,14787
18679,2020,Wyoming,Uinta,1591.0,16.82,7496.0,79.25,172.0,1.82,0.0,0.00,200.0,2.11,-62.43,96.07,9459
18680,2020,Wyoming,Washakie,651.0,16.15,3245.0,80.48,65.0,1.61,0.0,0.00,71.0,1.76,-64.33,96.63,4032


In [21]:
election_pivot.nsmallest(10, 'Two party vote share')

,Year,State,County,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Libertarian Votes,Libertarian Percentage,Green Votes,Green Percentage,Other Votes,Other Percentage,D-R margin,Two party vote share,Total
1797,2000,Missouri,Kansas City,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0
15205,2016,Utah,Cache,8563.0,18.55,21139.0,45.80,0.0,0.0,0.0,0.0,16455.0,35.65,-27.25,64.35,46157
13008,2016,Idaho,Madison,1201.0,7.66,8941.0,56.99,0.0,0.0,0.0,0.0,5546.0,35.35,-49.33,64.65,15688
15227,2016,Utah,Utah,28522.0,14.15,102182.0,50.70,0.0,0.0,0.0,0.0,70847.0,35.15,-36.55,64.85,201551
8206,2008,North Dakota,Grant,280.0,21.07,587.0,44.17,0.0,0.0,0.0,0.0,462.0,34.76,-23.10,65.24,1329
15208,2016,Utah,Davis,28776.0,20.79,62219.0,44.95,0.0,0.0,0.0,0.0,47416.0,34.26,-24.16,65.74,138411
7832,2008,Montana,Powder River,208.0,14.81,802.0,57.12,0.0,0.0,0.0,0.0,394.0,28.06,-42.31,71.93,1404
15225,2016,Utah,Tooele,4573.0,20.95,11169.0,51.17,0.0,0.0,0.0,0.0,6087.0,27.88,-30.22,72.12,21829
15217,2016,Utah,Morgan,577.0,11.11,3188.0,61.40,0.0,0.0,0.0,0.0,1427.0,27.48,-50.29,72.51,5192
15204,2016,Utah,Box Elder,2282.0,11.56,12230.0,61.97,0.0,0.0,0.0,0.0,5223.0,26.47,-50.41,73.53,19735


In some states in some years, certain minor parties did not file a candidate. However, it is suspicious if an entity has zero votes for either the Republican or Democratic parties. Let's drop those.

In [22]:
#This is just for debugging, can delete later
#election_pivot[election_pivot['State'].str.contains('Virginia')]

# Volatility
# For each county (i.e. State/County pair), find the average and total absolute swings between elections.
Add up the 2000-2004 swing, 2004-2008 swing, 2008-2012 swing, 2012-2016 swing, and 2016-2020 swing. This will identify *volatile* counties prone to large swings from year to year. A *volatile* county is not necessarily the same as a county that changes loyalty over time.

In [96]:
dupes = election_pivot[election_pivot.duplicated(subset=['County', 'State', 'Year'], keep=False)]

In [97]:
election_pivot = election_pivot.drop_duplicates(subset=['County', 'State', 'Year'])

In [98]:
election_pivot['CountyState'] = election_pivot['County'] + ', ' + election_pivot['State']
pivot = election_pivot.pivot(index='CountyState', columns='Year', values='D-R margin')

In [99]:
pivot

Year,2000,2004,2008,2012,2016,2020
CountyState,,,,,,
"Abbeville, South Carolina",-8.17,-10.55,-15.19,-13.48,-28.05,-33.08
"Acadia, Louisiana",-21.19,-28.33,-45.68,-49.82,-56.68,-60.34
"Accomack, Virginia",-10.57,-16.54,-1.45,-3.48,-11.65,-9.39
"Ada, Idaho",-27.86,-23.30,-6.18,-11.31,-9.23,-3.92
"Adair, Iowa",-12.66,-13.05,-3.36,-8.11,-34.85,-41.28
...,...,...,...,...,...,...
"Yuma, Arizona",-12.75,-15.99,-13.80,-12.61,-1.05,-6.19
"Yuma, Colorado",-47.59,-52.47,-48.38,-54.69,-65.21,-66.69
"Zapata, Texas",26.18,14.98,35.59,43.18,32.91,-5.35


In [100]:
pivot['2000-2004 swing'] = pivot[2004] - pivot[2000]
pivot['2000-2004 absolute swing'] = pivot['2000-2004 swing'].abs()
pivot['2004-2008 swing'] = pivot[2008] - pivot[2004]
pivot['2004-2008 absolute swing'] = pivot['2004-2008 swing'].abs()
pivot['2008-2012 swing'] = pivot[2012] - pivot[2008]
pivot['2008-2012 absolute swing'] = pivot['2008-2012 swing'].abs()
pivot['2012-2016 swing'] = pivot[2016] - pivot[2012]
pivot['2012-2016 absolute swing'] = pivot['2012-2016 swing'].abs()
pivot['2016-2020 swing'] = pivot[2020] - pivot[2016]
pivot['2016-2020 absolute swing'] = pivot['2016-2020 swing'].abs()
pivot['Total swing'] = pivot['2000-2004 swing'] + pivot['2004-2008 swing'] + pivot['2008-2012 swing'] + pivot['2012-2016 swing'] + pivot['2016-2020 swing']
pivot['Cumulative absolute swing'] = pivot['2000-2004 absolute swing'] + pivot['2004-2008 absolute swing'] + pivot['2008-2012 absolute swing'] + pivot['2012-2016 absolute swing'] + pivot['2016-2020 absolute swing']

In [101]:
pivot

Year,2000,2004,2008,2012,2016,2020,2000-2004 swing,2000-2004 absolute swing,2004-2008 swing,2004-2008 absolute swing,2008-2012 swing,2008-2012 absolute swing,2012-2016 swing,2012-2016 absolute swing,2016-2020 swing,2016-2020 absolute swing,Total swing,Cumulative absolute swing
CountyState,,,,,,,,,,,,,,,,,,
"Abbeville, South Carolina",-8.17,-10.55,-15.19,-13.48,-28.05,-33.08,-2.38,2.38,-4.64,4.64,1.71,1.71,-14.57,14.57,-5.03,5.03,-24.91,28.33
"Acadia, Louisiana",-21.19,-28.33,-45.68,-49.82,-56.68,-60.34,-7.14,7.14,-17.35,17.35,-4.14,4.14,-6.86,6.86,-3.66,3.66,-39.15,39.15
"Accomack, Virginia",-10.57,-16.54,-1.45,-3.48,-11.65,-9.39,-5.97,5.97,15.09,15.09,-2.03,2.03,-8.17,8.17,2.26,2.26,1.18,33.52
"Ada, Idaho",-27.86,-23.30,-6.18,-11.31,-9.23,-3.92,4.56,4.56,17.12,17.12,-5.13,5.13,2.08,2.08,5.31,5.31,23.94,34.20
"Adair, Iowa",-12.66,-13.05,-3.36,-8.11,-34.85,-41.28,-0.39,0.39,9.69,9.69,-4.75,4.75,-26.74,26.74,-6.43,6.43,-28.62,48.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Yuma, Arizona",-12.75,-15.99,-13.80,-12.61,-1.05,-6.19,-3.24,3.24,2.19,2.19,1.19,1.19,11.56,11.56,-5.14,5.14,6.56,23.32
"Yuma, Colorado",-47.59,-52.47,-48.38,-54.69,-65.21,-66.69,-4.88,4.88,4.09,4.09,-6.31,6.31,-10.52,10.52,-1.48,1.48,-19.10,27.28
"Zapata, Texas",26.18,14.98,35.59,43.18,32.91,-5.35,-11.20,11.20,20.61,20.61,7.59,7.59,-10.27,10.27,-38.26,38.26,-31.53,87.93


In [102]:
pivot['Cumulative absolute swing'].nlargest(10)

CountyState
Richmond, Virginia      107.97
Elliott, Kentucky       102.58
Starr, Texas            100.89
Maverick, Texas          99.21
Fairfax, Virginia        98.48
Jackson, Tennessee       96.43
Cache, Utah              96.01
Grundy, Tennessee        95.98
Mingo, West Virginia     93.05
Smith, Tennessee         92.96
Name: Cumulative absolute swing, dtype: float64

In [ ]:
election_pivot.nsmallest(10, 'Two party vote share')

To accomplish this, we will first create individual DataFrames for each election.

In [23]:
election2000 = election_pivot[election_pivot['Year'] == 2000]
election2004 = election_pivot[election_pivot['Year'] == 2004]
election2008 = election_pivot[election_pivot['Year'] == 2008]
election2012 = election_pivot[election_pivot['Year'] == 2012]
election2016 = election_pivot[election_pivot['Year'] == 2016]
election2020 = election_pivot[election_pivot['Year'] == 2020]

In [70]:
election2016

,Year,State,County,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Libertarian Votes,Libertarian Percentage,Green Votes,Green Percentage,Other Votes,Other Percentage,D-R margin,Two party vote share,Total
12455,2016,Alabama,Autauga,5936.0,23.77,18172.0,72.77,0.0,0.0,0.0,0.0,865.0,3.46,-49.00,96.54,24973
12456,2016,Alabama,Baldwin,18458.0,19.39,72883.0,76.55,0.0,0.0,0.0,0.0,3874.0,4.07,-57.16,95.94,95215
12457,2016,Alabama,Barbour,4871.0,46.53,5454.0,52.10,0.0,0.0,0.0,0.0,144.0,1.38,-5.57,98.63,10469
12458,2016,Alabama,Bibb,1874.0,21.25,6738.0,76.40,0.0,0.0,0.0,0.0,207.0,2.35,-55.15,97.65,8819
12459,2016,Alabama,Blount,2156.0,8.43,22859.0,89.33,0.0,0.0,0.0,0.0,573.0,2.24,-80.90,97.76,25588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15564,2016,Wyoming,Sweetwater,3231.0,18.86,12154.0,70.95,0.0,0.0,0.0,0.0,1745.0,10.19,-52.09,89.81,17130
15565,2016,Wyoming,Teton,7314.0,57.92,3921.0,31.05,0.0,0.0,0.0,0.0,1392.0,11.02,26.87,88.97,12627
15566,2016,Wyoming,Uinta,1202.0,14.19,6154.0,72.66,0.0,0.0,0.0,0.0,1114.0,13.15,-58.47,86.85,8470
15567,2016,Wyoming,Washakie,532.0,13.95,2911.0,76.32,0.0,0.0,0.0,0.0,371.0,9.73,-62.37,90.27,3814


# Predict 2000
Using actual data from 2004 to 2020

In [68]:
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

In [69]:
# Select relevant columns
target = 'Democrat Percentage'
features = [col for col in election_pivot.columns if col not in ['Year', 'State', 'County', target]]

In [70]:
# Split data into train (2004-2020) and test (2020)
train_data_rfr = election_pivot[(election_pivot['Year'] >= 2004) & (election_pivot['Year'] <= 2020)]
test_data_rfr = election_pivot[election_pivot['Year'] == 2000]

In [71]:
# Prepare X (features) and y (target)
X_train_rfr, y_train_rfr = train_data_rfr[features], train_data_rfr[target]
X_test_rfr, y_test_rfr = test_data_rfr[features], test_data_rfr[target]

In [72]:
# Train the model
rfr = RandomForestRegressor(random_state=42)
rfr.fit(X_train_rfr, y_train_rfr)

RandomForestRegressor(random_state=42)

In [73]:
# Predict for 2000
y_pred_rfr = rfr.predict(X_test_rfr)

In [74]:
# Convert the predictions to a DataFrame
y_pred_rfr_df = pd.DataFrame(y_pred_rfr, index=test_data_rfr.index, columns=['Predicted Democrat Percentage'])

# Include 'State' and 'County' columns in the comparison
comparison_df = pd.concat([test_data_rfr[['State', 'County', target]], y_pred_rfr_df], axis=1)

# Rename columns for clarity
comparison_df.columns = ['State', 'County', 'Actual Democrat Percentage', 'Predicted Democrat Percentage']

# Calculate the difference
comparison_df['Difference'] = comparison_df['Predicted Democrat Percentage'] - comparison_df['Actual Democrat Percentage']

# Display the comparison
comparison_df['Absolute Difference'] = comparison_df['Difference'].abs()

In [75]:
comparison_df

,State,County,Actual Democrat Percentage,Predicted Democrat Percentage,Difference,Absolute Difference
0,Alabama,Autauga,28.72,29.5213,0.8013,0.8013
1,Alabama,Baldwin,24.78,26.5406,1.7606,1.7606
2,Alabama,Barbour,49.91,49.5288,-0.3812,0.3812
3,Alabama,Bibb,38.16,38.8275,0.6675,0.6675
4,Alabama,Blount,27.69,28.5379,0.8479,0.8479
...,...,...,...,...,...,...
3108,Wyoming,Sweetwater,35.19,36.6974,1.5074,1.5074
3109,Wyoming,Teton,38.53,45.6593,7.1293,7.1293
3110,Wyoming,Uinta,22.24,23.3208,1.0808,1.0808
3111,Wyoming,Washakie,19.90,20.4699,0.5699,0.5699


In [76]:
comparison_df.nlargest(10, 'Absolute Difference')

,State,County,Actual Democrat Percentage,Predicted Democrat Percentage,Difference,Absolute Difference
1797,Missouri,Kansas City,0.00,92.5312,92.5312,92.5312
2535,Texas,Collin,24.45,48.1890,23.7390,23.7390
271,Colorado,San Miguel,49.09,65.9798,16.8898,16.8898
2756,Utah,Grand,32.04,47.1052,15.0652,15.0652
1598,Montana,Missoula,37.02,51.8456,14.8256,14.8256
179,California,Mendocino,48.34,62.9008,14.5608,14.5608
1193,Massachusetts,Hampshire,56.16,70.1932,14.0332,14.0332
1191,Massachusetts,Franklin,53.78,67.3776,13.5976,13.5976
270,Colorado,San Juan,34.17,47.3492,13.1792,13.1792
2865,Virginia,Richmond,101.29,88.1679,-13.1221,13.1221


# Predict 2020
Using actual data from 2000 to 2016

In [52]:
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

In [53]:
# Select relevant columns
target = 'Democrat Percentage'
features = [col for col in election_pivot.columns if col not in ['Year', 'State', 'County', target]]

In [54]:
# Split data into train (2000-2016) and test (2020)
train_data_rfr = election_pivot[(election_pivot['Year'] >= 2000) & (election_pivot['Year'] <= 2016)]
test_data_rfr = election_pivot[election_pivot['Year'] == 2020]

In [55]:
# Prepare X (features) and y (target)
X_train_rfr, y_train_rfr = train_data_rfr[features], train_data_rfr[target]
X_test_rfr, y_test_rfr = test_data_rfr[features], test_data_rfr[target]

In [56]:
# Train the model
rfr = RandomForestRegressor(random_state=42)
rfr.fit(X_train_rfr, y_train_rfr)

RandomForestRegressor(random_state=42)

In [57]:
# Predict for 2020
y_pred_rfr = rfr.predict(X_test_rfr)

In [60]:
# Convert the predictions to a DataFrame
y_pred_rfr_df = pd.DataFrame(y_pred_rfr, index=test_data_rfr.index, columns=['Predicted Democrat Percentage'])

# Include 'State' and 'County' columns in the comparison
comparison_df = pd.concat([test_data_rfr[['State', 'County', target]], y_pred_rfr_df], axis=1)

# Rename columns for clarity
comparison_df.columns = ['State', 'County', 'Actual Democrat Percentage', 'Predicted Democrat Percentage']

# Calculate the difference
comparison_df['Difference'] = comparison_df['Predicted Democrat Percentage'] - comparison_df['Actual Democrat Percentage']

# Display the comparison
comparison_df['Absolute Difference'] = comparison_df['Difference'].abs()

In [62]:
comparison_df.nlargest(10, 'Absolute Difference')

,State,County,Actual Democrat Percentage,Predicted Democrat Percentage,Difference,Absolute Difference
17746,Texas,Roberts,3.09,84.1834,81.0934,81.0934
15957,Georgia,Clayton,84.99,67.0155,-17.9745,17.9745
18280,South Dakota,Oglala Lakota,88.41,70.4653,-17.9447,17.9447
15970,Georgia,Dekalb,83.12,65.6127,-17.5073,17.5073
17482,Pennsylvania,Philadelphia,81.44,64.1513,-17.2887,17.2887
17566,Texas,Borden,3.85,17.3764,13.5264,13.5264
18284,South Dakota,Todd,77.31,64.4453,-12.8647,12.8647
17719,Texas,Montgomery,27.39,39.4275,12.0375,12.0375
16293,Indiana,Ohio,21.72,33.0060,11.2860,11.2860
18626,Wisconsin,Menominee,81.95,76.7956,-5.1544,5.1544


In [63]:
comparison_df.nsmallest(10, 'Absolute Difference')

,State,County,Actual Democrat Percentage,Predicted Democrat Percentage,Difference,Absolute Difference
15901,Florida,Martin,37.37,37.3710,0.0010,0.0010
15587,Alabama,Coosa,32.78,32.7786,-0.0014,0.0014
17692,Texas,Lavaca,13.07,13.0716,0.0016,0.0016
15616,Alabama,Marshall,14.99,14.9917,0.0017,0.0017
15605,Alabama,Jefferson,55.76,55.7618,0.0018,0.0018
15575,Alabama,Butler,41.79,41.7930,0.0030,0.0030
18292,Tennessee,Bedford,23.33,23.3344,0.0044,0.0044
15589,Alabama,Crenshaw,25.69,25.6847,-0.0053,0.0053
18293,Tennessee,Benton,21.06,21.0545,-0.0055,0.0055
17612,Texas,Dickens,13.16,13.1543,-0.0057,0.0057


In [61]:
comparison_df.nlargest(10, 'Difference')

,State,County,Actual Democrat Percentage,Predicted Democrat Percentage,Difference,Absolute Difference
17746,Texas,Roberts,3.09,84.1834,81.0934,81.0934
17566,Texas,Borden,3.85,17.3764,13.5264,13.5264
17719,Texas,Montgomery,27.39,39.4275,12.0375,12.0375
16293,Indiana,Ohio,21.72,33.0060,11.2860,11.2860
17684,Texas,King,5.03,10.0882,5.0582,5.0582
16122,Idaho,Madison,15.58,19.3808,3.8008,3.8008
18264,South Dakota,Lawrence,32.81,36.5429,3.7329,3.7329
18270,South Dakota,Meade,24.03,27.7356,3.7056,3.7056
17388,North Dakota,Steele,36.03,39.6500,3.6200,3.6200
18410,Utah,Utah,26.75,30.1864,3.4364,3.4364


In [56]:
comparison_df.nsmallest(10, 'Difference')

,State,County,Actual Democrat Percentage,Predicted Democrat Percentage,Difference
17497,Pennsylvania,Philadelphia,81.44,77.0012,-4.4388
15972,Georgia,Clayton,84.99,80.7399,-4.2501
15985,Georgia,Dekalb,83.12,79.0401,-4.0799
18296,South Dakota,Oglala Lakota,88.41,84.7742,-3.6358
16011,Georgia,Hancock,71.66,68.1065,-3.5535
16063,Georgia,Rockdale,69.92,66.8806,-3.0394
15988,Georgia,Dougherty,69.60,66.5625,-3.0375
16001,Georgia,Fulton,72.65,69.6637,-2.9863
15872,Delaware,New Castle,67.81,65.1215,-2.6885
18300,South Dakota,Todd,77.31,74.6276,-2.6824


In [57]:
(comparison_df['Difference'] > 0).value_counts()

Difference
True     2968
False     146
Name: count, dtype: int64

# Take averages and aggregates by state

For each state, find the average absolute error by each county.

Make a spikeplot similar to my climate project.

In [ ]:
# Prepare X (features) and y (target)
X_train_rfr, y_train_rfr = train_data_rfr[features], train_data_rfr[target]
X_test_rfr, y_test_rfr = test_data_rfr[features], test_data_rfr[target]

In [ ]:
rfr_params = {
    'n_estimators': [50, 100, 200, 300],  # Number of trees
    'max_depth': [None, 10, 20, 30],  # Maximum depth of trees
    'min_samples_split': [2, 5, 10],  # Minimum samples per split
    'min_samples_leaf': [1, 2, 4],  # Minimum samples per leaf
    'max_features': ['sqrt', 'log2']  # Number of features to consider for splits
}

In [ ]:
rfr = RandomForestRegressor(random_state=42)

In [ ]:
rfr_gs = GridSearchCV(estimator=rfr, param_grid=rfr_params, 
                           cv=5, scoring='neg_mean_squared_error', 
                           n_jobs=-1, verbose=1)

In [ ]:
%%time
# Fit on training data
rfr_gs.fit(X_train_rfr, y_train_rfr)

In [209]:
import pandas as pd

# Assuming your DataFrame is named df
df_pivoted_votes = elections.pivot_table(index='County', columns=['Year', 'Party'], values='Votes', aggfunc='sum')
df_pivoted_percentage = elections.pivot_table(index='County', columns=['Year', 'Party'], values='Percentage', aggfunc='sum')

# Flatten the multi-level column index
df_pivoted_votes.columns = [f"{year} {party} Votes" for year, party in df_pivoted_votes.columns]
df_pivoted_percentage.columns = [f"{year} {party} Percentage" for year, party in df_pivoted_percentage.columns]

# Combine the two DataFrames
df_final = pd.concat([df_pivoted_votes, df_pivoted_percentage], axis=1)

# Sort by County if needed
df_final = df_final.sort_index(axis=0)

# Display the final result
df_final

,2000 Democrat Votes,2000 Green Votes,2000 Other Votes,2000 Republican Votes,2004 Democrat Votes,2004 Other Votes,2004 Republican Votes,2008 Democrat Votes,2008 Other Votes,2008 Republican Votes,...,2012 Other Percentage,2012 Republican Percentage,2016 Democrat Percentage,2016 Other Percentage,2016 Republican Percentage,2020 Democrat Percentage,2020 Green Percentage,2020 Libertarian Percentage,2020 Other Percentage,2020 Republican Percentage
County,,,,,,,,,,,,,,,,,,,,,
"Abbeville, South Carolina",3766.0,66.0,92.0,4450.0,4389.0,100.0,5436.0,4593.0,144.0,6264.0,...,1.38,56.05,34.72,2.52,62.77,32.98,0.18,0.71,0.05,66.06
"Acadia, Louisiana",8892.0,227.0,305.0,13814.0,8937.0,210.0,16083.0,7028.0,454.0,19228.0,...,1.28,74.27,20.58,2.15,77.26,19.15,NaN,0.85,0.51,79.49
"Accomack, Virginia",5092.0,220.0,261.0,6352.0,5518.0,112.0,7726.0,7607.0,183.0,7833.0,...,1.14,51.17,42.61,3.13,54.26,44.68,NaN,1.11,0.14,54.07
"Ada, Idaho",40650.0,4846.0,2939.0,75050.0,58523.0,1838.0,94641.0,82236.0,3838.0,93328.0,...,3.24,54.03,38.68,13.41,47.91,46.47,NaN,2.05,1.10,50.39
"Adair, Iowa",1753.0,66.0,29.0,2275.0,1844.0,32.0,2402.0,1924.0,69.0,2060.0,...,2.30,52.90,29.73,5.69,64.58,28.68,0.09,0.84,0.43,69.96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Yuma, Arizona",12055.0,495.0,394.0,15708.0,16032.0,249.0,22184.0,18559.0,479.0,24577.0,...,1.29,55.66,46.42,6.11,47.47,46.09,0.00,1.63,0.00,52.28
"Yuma, Colorado",1082.0,60.0,60.0,3156.0,1064.0,39.0,3456.0,1117.0,80.0,3286.0,...,2.18,76.25,15.15,4.49,80.36,15.76,0.04,1.04,0.70,82.45
"Zapata, Texas",1638.0,11.0,14.0,953.0,1662.0,8.0,1228.0,1939.0,8.0,919.0,...,0.54,28.14,65.66,1.59,32.75,47.13,0.10,0.28,0.00,52.48


In [6]:
import pandas as pd

# Assuming your dataframe is called 'election_data'
pivoted_df = elections.pivot_table(index=['county_name'], columns=['year', 'party'], values='candidatevotes', aggfunc='sum')

# Flatten the MultiIndex columns (if necessary)
pivoted_df.columns = [f'{party}_{year}' for year, party in pivoted_df.columns]

# Reset the index if you want 'County' as a column
pivoted_df.reset_index(inplace=True)

# Display the reformatted dataframe
pivoted_df

,county_name,DEMOCRAT_2000,GREEN_2000,OTHER_2000,REPUBLICAN_2000,DEMOCRAT_2004,OTHER_2004,REPUBLICAN_2004,DEMOCRAT_2008,OTHER_2008,...,OTHER_2012,REPUBLICAN_2012,DEMOCRAT_2016,OTHER_2016,REPUBLICAN_2016,DEMOCRAT_2020,GREEN_2020,LIBERTARIAN_2020,OTHER_2020,REPUBLICAN_2020
0,ABBEVILLE,3766.0,66.0,92.0,4450.0,4389.0,100.0,5436.0,4593.0,144.0,...,147.0,5981.0,3741.0,271.0,6763.0,4101.0,23.0,88.0,6.0,8215.0
1,ACADIA,8892.0,227.0,305.0,13814.0,8937.0,210.0,16083.0,7028.0,454.0,...,344.0,19931.0,5638.0,589.0,21162.0,5443.0,NaN,241.0,145.0,22596.0
2,ACCOMACK,5092.0,220.0,261.0,6352.0,5518.0,112.0,7726.0,7607.0,183.0,...,183.0,8213.0,6740.0,495.0,8583.0,7578.0,NaN,188.0,24.0,9172.0
3,ADA,40650.0,4846.0,2939.0,75050.0,58523.0,1838.0,94641.0,82236.0,3838.0,...,5855.0,97554.0,75677.0,26240.0,93752.0,120539.0,NaN,5310.0,2841.0,130699.0
4,ADAIR,9994.0,415.0,283.0,17288.0,11108.0,186.0,19368.0,11379.0,432.0,...,447.0,17987.0,7338.0,1530.0,19915.0,7687.0,47.0,314.0,140.0,22196.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1887,YUBA,5546.0,507.0,236.0,9838.0,5687.0,257.0,12076.0,8866.0,528.0,...,714.0,11275.0,7910.0,1918.0,13170.0,11230.0,124.0,510.0,256.0,17676.0
1888,YUMA,13137.0,555.0,454.0,18864.0,17096.0,288.0,25640.0,19676.0,559.0,...,642.0,26842.0,25331.0,3455.0,29015.0,32995.0,2.0,1189.0,35.0,40641.0
1889,ZAPATA,1638.0,11.0,14.0,953.0,1662.0,8.0,1228.0,1939.0,8.0,...,19.0,997.0,2063.0,50.0,1029.0,1826.0,4.0,11.0,0.0,2033.0
1890,ZAVALA,2616.0,14.0,10.0,751.0,2332.0,9.0,777.0,3263.0,17.0,...,37.0,574.0,2636.0,66.0,694.0,2864.0,12.0,13.0,0.0,1490.0
